# Outputs Test Notebook

This notebook tests notebook output rendering in JupyterLab environments. Each cell exercises a
specific output type or behavior from the Jupyter messaging protocol and nbformat spec.

Use this to verify that outputs are correctly:
- Captured from the kernel
- Written to the YDoc (collaborative model)
- Rendered in the frontend
- Persisted to disk on save
- Restored on notebook reload

In [ ]:
%pip install numpy matplotlib ipywidgets

## Basic Outputs

### Test: Simple stdout
**Output type:** `stream` (name: `stdout`)

The most basic output — a single `print()` call producing one stream message.

In [ ]:
print("Hello, world!")

### Test: stderr stream
**Output type:** `stream` (name: `stderr`)

Verifies that stderr is rendered distinctly from stdout (typically in red/different styling).

In [ ]:
import sys
print("This is stdout")
print("This is stderr", file=sys.stderr)
print("Back to stdout")

### Test: execute_result
**Output type:** `execute_result`

When the last expression in a cell has a value, the kernel sends an `execute_result` message
(distinct from `display_data`). JupyterLab renders this with an `Out[n]:` prompt.

In [ ]:
{'answer': 42, 'pi': 3.14159, 'greeting': 'hello'}

### Test: Error traceback
**Output type:** `error`

Verifies that tracebacks are rendered with ANSI color codes and proper formatting.

In [ ]:
def outer():
    def inner():
        raise ValueError("Something went wrong!")
    inner()

outer()

## Streaming & Dynamic Outputs

### Test: Streaming output
**Output type:** `stream` (incremental)

Prints lines one per second. Tests that stream outputs are appended incrementally
rather than buffered until cell completion.

In [ ]:
import asyncio

for i in range(1, 21):
    print(f"Count: {i}/20", flush=True)
    await asyncio.sleep(1)

### Test: display_id update (in-place progress)
**Output type:** `display_data` + `update_display_data`

Creates a display with a `display_id`, then updates it in-place 10 times.
Tests that `update_display_data` correctly replaces the output at the tracked index.

In [ ]:
from IPython.display import display, update_display
import asyncio

handle = display("Starting...", display_id="progress")

for i in range(1, 11):
    await asyncio.sleep(0.5)
    update_display(f"Progress: {'█' * i}{'░' * (10 - i)} {i*10}%", display_id="progress")

update_display("✅ Complete!", display_id="progress")

### Test: clear_output (wait=True)
**Output type:** `clear_output`

Uses `clear_output(wait=True)` to replace output frame-by-frame. The `wait=True` flag
means the clear is deferred until the next output arrives, preventing flicker.

In [ ]:
from IPython.display import clear_output
import asyncio

for i in range(10):
    clear_output(wait=True)
    print(f"Frame {i}: {'*' * (i+1)}")
    await asyncio.sleep(0.5)

clear_output(wait=True)
print("Animation complete!")

## Rich Display Outputs

### Test: HTML display
**Output type:** `display_data` (mime: `text/html`)

Renders styled HTML content. Tests the HTML mime renderer.

In [ ]:
from IPython.display import display, HTML

display(HTML("""
<div style="padding: 12px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 8px; color: white;">
    <h3 style="margin: 0;">Rich HTML Output</h3>
    <p style="margin: 8px 0 0 0;">This tests styled HTML rendering in the output area.</p>
</div>
"""))

### Test: LaTeX / Math rendering
**Output type:** `display_data` (mime: `text/latex`)

Renders LaTeX mathematical expressions. Tests the MathJax/KaTeX renderer.

In [ ]:
from IPython.display import display, Math, Latex

display(Math(r'\int_{-\infty}^{\infty} e^{-x^2} dx = \sqrt{\pi}'))
display(Math(r'\nabla \times \mathbf{E} = -\frac{\partial \mathbf{B}}{\partial t}'))
display(Latex(r"""
\begin{align}
E &= mc^2 \\
F &= ma \\
PV &= nRT
\end{align}
"""))

### Test: SVG output
**Output type:** `display_data` (mime: `image/svg+xml`)

Renders inline SVG graphics. Tests the SVG mime renderer.

In [ ]:
from IPython.display import SVG, display

svg = """
<svg width="300" height="200" xmlns="http://www.w3.org/2000/svg">
  <defs>
    <linearGradient id="grad" x1="0%" y1="0%" x2="100%" y2="100%">
      <stop offset="0%" style="stop-color:#4CAF50;stop-opacity:1" />
      <stop offset="100%" style="stop-color:#2196F3;stop-opacity:1" />
    </linearGradient>
  </defs>
  <rect x="10" y="10" width="280" height="180" rx="15" fill="url(#grad)" />
  <circle cx="80" cy="100" r="40" fill="rgba(255,255,255,0.3)" />
  <circle cx="150" cy="80" r="25" fill="rgba(255,255,255,0.2)" />
  <text x="150" y="140" text-anchor="middle" fill="white" font-size="20" font-family="sans-serif">SVG Output</text>
</svg>
"""
display(SVG(svg))

### Test: Multiple mime types in one output
**Output type:** `display_data` (multiple mimes)

A single output can contain multiple representations (text/plain, text/html, image/png, etc.).
JupyterLab picks the richest available renderer. This tests that mime priority works correctly.

In [ ]:
class MultiMimeObject:
    """An object that provides multiple representations."""
    def __repr__(self):
        return "MultiMimeObject(plain text fallback)"
    
    def _repr_html_(self):
        return '<div style="padding:8px; border:2px solid #4CAF50; border-radius:4px;"><b>HTML representation</b> (preferred over plain text)</div>'
    
    def _repr_latex_(self):
        return r'$\text{MultiMimeObject}_{\LaTeX}$'
    
    def _repr_json_(self):
        return {"type": "MultiMimeObject", "renderers": ["plain", "html", "latex", "json"]}

MultiMimeObject()

## Widgets

### Test: Interactive widgets (ipywidgets)
**Output type:** `display_data` (mime: `application/vnd.jupyter.widget-view+json`)

Renders an interactive slider widget with a live callback. Tests the widget comm protocol
and `application/vnd.jupyter.widget-view+json` mime type.

> **Note:** Requires a server restart after installing ipywidgets for the labextension to register.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

slider = widgets.IntSlider(value=50, min=0, max=100, description='Value:')
output = widgets.Output()

def on_change(change):
    with output:
        output.clear_output()
        print(f"Slider value: {change['new']}")

slider.observe(on_change, names='value')
display(slider, output)

## Plots & Visualizations

### Test: Matplotlib plots
**Output type:** `display_data` (mime: `image/png`)

Generates static PNG plots via matplotlib. Tests image output rendering and
the handling of potentially large base64-encoded image data.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(0, 10, 100)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(x, np.sin(x), 'b-', label='sin(x)')
ax1.plot(x, np.cos(x), 'r--', label='cos(x)')
ax1.set_title('Trigonometric Functions')
ax1.legend()

ax2.bar(['A', 'B', 'C', 'D'], [23, 45, 12, 67], color=['#4C72B0', '#55A868', '#C44E52', '#8172B2'])
ax2.set_title('Bar Chart')

plt.tight_layout()
plt.show()

### (Plotly test omitted — requires jupyterlab-plotly extension)